# EDA de GitHub Agentic Workflows

Este notebook documenta el origen, el esquema y la calidad del dataset propio de chikiMiner. Se ejecuta de forma independiente y no depende del estado de otro notebook.

El análisis usa el snapshot local de huggingface-dataset/. El dataset propio está publicado en https://huggingface.co/datasets/SebaS01010101/GHAW-H. No se utiliza el dataset temporal de pavtch/GHAW-H.

# 1. Origen y carga de los datos

## 1.1 Dataset utilizado

La fuente es el dataset propio producido por la Tarea 3 y publicado en https://huggingface.co/datasets/SebaS01010101/GHAW-H. El notebook trabaja con la copia local para reproducir exactamente el snapshot.

**No se utiliza el dataset temporal de pavtch/GHAW-H**.

## 1.2 Importación de librerías

## 1.3 Configuración de rutas

La ruta puede sobreescribirse con CHIKIMINER_EDA_DATA_DIR. Si no existe, se busca huggingface-dataset/ en el directorio de ejecución y en sus directorios padre.

## 1.4 Carga de Parquet

Se validan los cuatro archivos antes de cargarlos.

## 1.5 Descripción de cada tabla

In [1]:
from pathlib import Path
import json
import re
import os

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Markdown, display

TABLE_FILES = {
    'repositories': 'repositories.parquet',
    'workflow_files': 'workflow_files.parquet',
    'frontmatter_fields': 'frontmatter_fields.parquet',
    'workflow_bodies': 'workflow_bodies.parquet',
}

def find_dataset_dir():
    candidates = []
    configured = os.environ.get('CHIKIMINER_EDA_DATA_DIR')
    if configured:
        candidates.append(Path(configured).expanduser())
    current = Path.cwd()
    candidates.extend([current / 'huggingface-dataset', current.parent / 'huggingface-dataset'])
    candidates.extend(parent / 'huggingface-dataset' for parent in current.parents)
    for candidate in candidates:
        candidate = candidate.resolve()
        if all((candidate / filename).is_file() for filename in TABLE_FILES.values()):
            return candidate
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError('No se encontraron los cuatro Parquet requeridos. Rutas revisadas:\n' + searched)

DATA_DIR = find_dataset_dir()
PARQUET_PATHS = {name: DATA_DIR / filename for name, filename in TABLE_FILES.items()}
missing = [str(path) for path in PARQUET_PATHS.values() if not path.is_file()]
assert not missing, f'Faltan archivos Parquet: {missing}'

tables = {name: pd.read_parquet(path) for name, path in PARQUET_PATHS.items()}
repositories = tables['repositories']
workflow_files = tables['workflow_files']
frontmatter_fields = tables['frontmatter_fields']
workflow_bodies = tables['workflow_bodies']

table_names = ', '.join(tables)
display(Markdown('**Directorio de datos:** ' + str(DATA_DIR) + '\n\n**Tablas cargadas:** ' + table_names + '\n\n**Hugging Face:** https://huggingface.co/datasets/SebaS01010101/GHAW-H; se usa el dataset propio local.'))

**Directorio de datos:** C:\Users\vicen\OneDrive\Documentos\GitHub\Ing de Datos\ChikiMiner\huggingface-dataset

**Tablas cargadas:** repositories, workflow_files, frontmatter_fields, workflow_bodies

**Hugging Face:** https://huggingface.co/datasets/SebaS01010101/GHAW-H; se usa el dataset propio local.

In [2]:
def pandas_type_summary(frame):
    return '; '.join(f'{column}: {dtype}' for column, dtype in frame.dtypes.items())

overview = pd.DataFrame([
    {
        'table': name,
        'path': str(PARQUET_PATHS[name]),
        'rows': len(frame),
        'columns': len(frame.columns),
        'memory_mb': round(frame.memory_usage(deep=True).sum() / 1024**2, 3),
        'pandas_types': pandas_type_summary(frame),
    }
    for name, frame in tables.items()
])
display(overview)

,table,path,rows,columns,memory_mb,pandas_types
0,repositories,C:\Users\vicen\OneDrive\Documentos\GitHub\Ing ...,374,6,0.153,repository_id: object; full_name: object; cano...
1,workflow_files,C:\Users\vicen\OneDrive\Documentos\GitHub\Ing ...,1541,10,13.953,file_id: object; repository_id: object; filena...
2,frontmatter_fields,C:\Users\vicen\OneDrive\Documentos\GitHub\Ing ...,60038,5,34.917,field_id: object; file_id: object; key_path: o...
3,workflow_bodies,C:\Users\vicen\OneDrive\Documentos\GitHub\Ing ...,1541,2,31.059,file_id: object; body_markdown: object


# 2. Descripción de las tablas y sus relaciones

La unidad de fila se determina a partir del código de Tarea 3 y se contrasta con las claves observadas. frontmatter_fields tiene una fila por nodo YAML normalizado, no una fila por archivo. Por eso sus 60.038 filas no representan 60.038 archivos.

Relaciones conceptuales:

- repositories 1 → N workflow_files.
- workflow_files 1 → N frontmatter_fields.
- workflow_files 1 → 1 workflow_bodies.

In [3]:
PRIMARY_KEYS = {
    'repositories': ['repository_id'],
    'workflow_files': ['file_id'],
    'frontmatter_fields': ['field_id'],
    'workflow_bodies': ['file_id'],
}
FOREIGN_KEYS = {
    'repositories': '—',
    'workflow_files': 'repository_id → repositories.repository_id',
    'frontmatter_fields': 'file_id → workflow_files.file_id',
    'workflow_bodies': 'file_id → workflow_files.file_id (también PK)',
}
ROW_UNITS = {
    'repositories': '1 fila = 1 repositorio',
    'workflow_files': '1 fila = 1 archivo Markdown',
    'frontmatter_fields': '1 fila = 1 nodo YAML normalizado',
    'workflow_bodies': '1 fila = body Markdown de 1 archivo',
}

schema_rows = []
for name, frame in tables.items():
    arrow_schema = pq.read_schema(PARQUET_PATHS[name])
    arrow_types = '; '.join(f'{field.name}: {field.type}' for field in arrow_schema)
    schema_rows.append({
        'table': name,
        'rows': len(frame),
        'columns': len(frame.columns),
        'pyarrow_types': arrow_types,
        'pandas_types': pandas_type_summary(frame),
        'memory_mb': round(frame.memory_usage(deep=True).sum() / 1024**2, 3),
        'primary_key': ', '.join(PRIMARY_KEYS[name]),
        'foreign_keys': FOREIGN_KEYS[name],
        'row_unit': ROW_UNITS[name],
    })
schema_summary = pd.DataFrame(schema_rows)
display(schema_summary)

,table,rows,columns,pyarrow_types,pandas_types,memory_mb,primary_key,foreign_keys,row_unit
0,repositories,374,6,repository_id: string; full_name: string; cano...,repository_id: object; full_name: object; cano...,0.153,repository_id,—,1 fila = 1 repositorio
1,workflow_files,1541,10,file_id: string; repository_id: string; filena...,file_id: object; repository_id: object; filena...,13.953,file_id,repository_id → repositories.repository_id,1 fila = 1 archivo Markdown
2,frontmatter_fields,60038,5,field_id: string; file_id: string; key_path: s...,field_id: object; file_id: object; key_path: o...,34.917,field_id,file_id → workflow_files.file_id,1 fila = 1 nodo YAML normalizado
3,workflow_bodies,1541,2,file_id: string; body_markdown: large_string,file_id: object; body_markdown: object,31.059,file_id,file_id → workflow_files.file_id (también PK),1 fila = body Markdown de 1 archivo


In [4]:
repo_unique = repositories['repository_id'].nunique()
file_unique = workflow_files['file_id'].nunique()
field_unique = frontmatter_fields['field_id'].nunique()
body_unique = workflow_bodies['file_id'].nunique()
repo_file_counts = workflow_files.groupby('repository_id').size().reindex(repositories['repository_id'], fill_value=0)
field_counts = frontmatter_fields.groupby('file_id').size()

relationship_summary = pd.DataFrame([
    {'relationship': 'repositories → workflow_files', 'expected': '1:N', 'observed': str(repo_unique) + ' repos / ' + str(file_unique) + ' files', 'key': 'repository_id'},
    {'relationship': 'workflow_files → frontmatter_fields', 'expected': '1:N', 'observed': str(file_unique) + ' files / ' + str(field_unique) + ' fields; ' + str(field_counts.size) + ' files con al menos un campo', 'key': 'file_id'},
    {'relationship': 'workflow_files → workflow_bodies', 'expected': '1:1', 'observed': str(file_unique) + ' files / ' + str(body_unique) + ' bodies', 'key': 'file_id'},
])
display(relationship_summary)
display(pd.DataFrame({
    'metric': ['repositorios únicos', 'repositorios con archivos', 'repositorios sin archivos', 'archivos Markdown únicos', 'frontmatter fields', 'bodies'],
    'value': [repo_unique, int((repo_file_counts > 0).sum()), int((repo_file_counts == 0).sum()), file_unique, field_unique, body_unique],
}))
repositories_without_files = repositories.loc[repositories['repository_id'].map(repo_file_counts).fillna(0).eq(0), ['repository_id', 'full_name']].copy()
repositories_without_files['n_files'] = 0
display(repositories_without_files)

,relationship,expected,observed,key
0,repositories → workflow_files,1:N,374 repos / 1541 files,repository_id
1,workflow_files → frontmatter_fields,1:N,1541 files / 60038 fields; 1505 files con al m...,file_id
2,workflow_files → workflow_bodies,1:1,1541 files / 1541 bodies,file_id


,metric,value
0,repositorios únicos,374
1,repositorios con archivos,369
2,repositorios sin archivos,5
3,archivos Markdown únicos,1541
4,frontmatter fields,60038
5,bodies,1541


,repository_id,full_name,n_files
48,6559dff0e9ab463f5761c2abdbf7c921f9f91bee050a9b...,BMayhew/awesome-sites-to-test-on,0
273,70a418ea2f61bdcb3f2a3a4cc4641d4efccafea3a6516d...,neurodesk/neurodesktop,0
293,9ab29c944a39a32c64878124e14653b23700301abe2d4a...,pikax/verter,0
333,711d26e018e99081364d03363944cf806161cd4b87869e...,SocketDev/action,0
341,5bc8de7601de0e1011d3d7fcc2af7b254e560a4f2ae3c9...,stride3d/stride-community-toolkit,0


# 3. Revisión de calidad

Los controles siguientes son diagnósticos. No modifican los Parquet originales ni eliminan registros. La columna status distingue una condición válida, una observación que debe documentarse y un posible problema estructural.

La ausencia de frontmatter no se interpreta automáticamente como error: 36 archivos tienen parse_status = no_frontmatter y frontmatter_raw = NULL, un caso compatible con archivos Markdown sin bloque YAML opcional.

In [5]:
quality_rows = []

def add_control(control, table, affected_rows, status, observation):
    quality_rows.append({
        'control': control,
        'table': table,
        'affected_rows': int(affected_rows),
        'status': status,
        'observation': observation,
    })

for name, frame in tables.items():
    full_duplicates = int(frame.duplicated().sum())
    add_control('duplicados completos', name, full_duplicates, 'OK' if full_duplicates == 0 else 'REVISAR', 'No se detectaron filas idénticas.' if full_duplicates == 0 else 'Existen filas idénticas.')
    for key in PRIMARY_KEYS[name]:
        nulls = int(frame[key].isna().sum())
        empties = int(frame[key].astype('string').fillna('').str.strip().eq('').sum())
        duplicates = int(frame[key].duplicated(keep=False).sum())
        add_control('PK nula: ' + key, name, nulls, 'OK' if nulls == 0 else 'REVISAR', 'No hay claves primarias nulas.' if nulls == 0 else 'Hay claves primarias nulas.')
        add_control('PK vacía: ' + key, name, empties, 'OK' if empties == 0 else 'REVISAR', 'No hay claves primarias vacías.' if empties == 0 else 'Hay claves primarias vacías.')
        add_control('PK duplicada: ' + key, name, duplicates, 'OK' if duplicates == 0 else 'REVISAR', 'La PK es única.' if duplicates == 0 else 'Hay valores de PK repetidos.')

orphan_repositories = int((~workflow_files['repository_id'].isin(repositories['repository_id'])).sum())
orphan_fields = int((~frontmatter_fields['file_id'].isin(workflow_files['file_id'])).sum())
orphan_bodies = int((~workflow_bodies['file_id'].isin(workflow_files['file_id'])).sum())
files_without_body = int((~workflow_files['file_id'].isin(workflow_bodies['file_id'])).sum())
bodies_per_file = workflow_bodies.groupby('file_id').size()
multiple_bodies = int((bodies_per_file > 1).sum())
add_control('FK repository_id huérfana', 'workflow_files', orphan_repositories, 'OK' if orphan_repositories == 0 else 'REVISAR', 'Todas las FK apuntan a repositories.' if orphan_repositories == 0 else 'Hay archivos sin repositorio.')
add_control('FK file_id huérfana', 'frontmatter_fields', orphan_fields, 'OK' if orphan_fields == 0 else 'REVISAR', 'Todos los campos apuntan a workflow_files.' if orphan_fields == 0 else 'Hay campos sin archivo.')
add_control('FK file_id huérfana', 'workflow_bodies', orphan_bodies, 'OK' if orphan_bodies == 0 else 'REVISAR', 'Todos los bodies apuntan a workflow_files.' if orphan_bodies == 0 else 'Hay bodies sin archivo.')
add_control('archivos sin body', 'workflow_files', files_without_body, 'OK' if files_without_body == 0 else 'REVISAR', 'La relación archivo-body está completa.' if files_without_body == 0 else 'Hay archivos sin body.')
add_control('múltiples bodies por file_id', 'workflow_bodies', multiple_bodies, 'OK' if multiple_bodies == 0 else 'REVISAR', 'Hay como máximo un body por archivo.' if multiple_bodies == 0 else 'Hay archivos con varios bodies.')

invalid_json = 0
for raw in frontmatter_fields['value_json']:
    try:
        json.loads(raw)
    except (TypeError, json.JSONDecodeError):
        invalid_json += 1
add_control('JSON inválido en value_json', 'frontmatter_fields', invalid_json, 'OK' if invalid_json == 0 else 'REVISAR', 'Todos los valores se pueden decodificar con json.loads().' if invalid_json == 0 else 'Hay valores no decodificables.')

parse_errors = int(workflow_files['parse_error'].notna().sum())
add_control('parse_error no nulo', 'workflow_files', parse_errors, 'OK' if parse_errors == 0 else 'REVISAR', 'No se registraron errores de parseo.' if parse_errors == 0 else 'Hay errores de parseo registrados.')

display(pd.DataFrame(quality_rows))

,control,table,affected_rows,status,observation
0,duplicados completos,repositories,0,OK,No se detectaron filas idénticas.
1,PK nula: repository_id,repositories,0,OK,No hay claves primarias nulas.
2,PK vacía: repository_id,repositories,0,OK,No hay claves primarias vacías.
3,PK duplicada: repository_id,repositories,0,OK,La PK es única.
4,duplicados completos,workflow_files,0,OK,No se detectaron filas idénticas.
5,PK nula: file_id,workflow_files,0,OK,No hay claves primarias nulas.
6,PK vacía: file_id,workflow_files,0,OK,No hay claves primarias vacías.
7,PK duplicada: file_id,workflow_files,0,OK,La PK es única.
8,duplicados completos,frontmatter_fields,0,OK,No se detectaron filas idénticas.
9,PK nula: field_id,frontmatter_fields,0,OK,No hay claves primarias nulas.


In [6]:
null_rows = []
empty_string_rows = []
for name, frame in tables.items():
    for column in frame.columns:
        null_count = int(frame[column].isna().sum())
        null_rows.append({'table': name, 'column': column, 'null_rows': null_count, 'percentage': round(100 * null_count / len(frame), 3) if len(frame) else 0})
        if pd.api.types.is_object_dtype(frame[column]) or pd.api.types.is_string_dtype(frame[column]):
            series = frame[column].astype('string')
            empty_count = int((series.notna() & series.str.strip().eq('')).sum())
            if empty_count:
                empty_string_rows.append({'table': name, 'column': column, 'empty_or_whitespace_non_null': empty_count})

null_summary = pd.DataFrame(null_rows).query('null_rows > 0').sort_values(['table', 'null_rows'], ascending=[True, False])
display(Markdown('**Valores nulos observados**'))
display(null_summary if not null_summary.empty else pd.DataFrame({'observation': ['No hay valores nulos.']}))
display(Markdown('**Strings vacíos o con whitespace, excluyendo NULL**'))
display(pd.DataFrame(empty_string_rows) if empty_string_rows else pd.DataFrame({'observation': ['No hay strings vacíos no nulos.']}))

parse_status_summary = workflow_files['parse_status'].value_counts(dropna=False).rename_axis('parse_status').reset_index(name='files')
frontmatter_absence = pd.DataFrame([
    {'condition': 'parse_status = no_frontmatter', 'files': int(workflow_files['parse_status'].eq('no_frontmatter').sum()), 'interpretation': 'Ausencia opcional de bloque YAML; no se clasifica automáticamente como corrupción.'},
    {'condition': 'frontmatter_raw NULL', 'files': int(workflow_files['frontmatter_raw'].isna().sum()), 'interpretation': 'Coincide con los archivos sin frontmatter observados.'},
])
display(parse_status_summary)
display(frontmatter_absence)

**Valores nulos observados**

,table,column,null_rows,percentage
14,workflow_files,parse_error,1541,100.000
11,workflow_files,frontmatter_raw,36,2.336


**Strings vacíos o con whitespace, excluyendo NULL**

,table,column,empty_or_whitespace_non_null
0,workflow_bodies,body_markdown,1


,parse_status,files
0,ok,1505
1,no_frontmatter,36


,condition,files,interpretation
0,parse_status = no_frontmatter,36,Ausencia opcional de bloque YAML; no se clasif...
1,frontmatter_raw NULL,36,Coincide con los archivos sin frontmatter obse...


## 3.1 Paths YAML escapados y tipos

Los paths normalizados pueden contener puntos escapados. El helper siguiente busca el primer punto que no esté precedido por una barra invertida; por tanto ms\.date permanece como una raíz completa. No se usa str.split('.').

In [7]:
def first_unescaped_dot(path):
    escaped = False
    for index, character in enumerate(str(path)):
        if character == chr(92) and not escaped:
            escaped = True
            continue
        if character == '.' and not escaped:
            return index
        escaped = False
    return -1

def root_key(path):
    path = str(path)
    separator = first_unescaped_dot(path)
    return path if separator == -1 else path[:separator]

assert root_key('permissions.contents') == 'permissions'
assert root_key('tools.github.toolsets') == 'tools'
assert root_key('ms' + chr(92) + '.date') == 'ms' + chr(92) + '.date'
assert root_key('metadata.date') == 'metadata'

field_profile = frontmatter_fields.copy()
field_profile['root_key'] = field_profile['key_path'].map(root_key)
field_profile['depth'] = field_profile['key_path'].map(lambda value: sum(1 for i, char in enumerate(str(value)) if char == '.' and (i == 0 or str(value)[i - 1] != chr(92))) + 1)

def decoded_json_kind(raw):
    value = json.loads(raw)
    if value is None:
        return 'null'
    if isinstance(value, bool):
        return 'boolean'
    if isinstance(value, dict):
        return 'mapping'
    if isinstance(value, list):
        return 'array'
    if isinstance(value, int):
        return 'integer'
    if isinstance(value, float):
        return 'float'
    return 'string'

field_profile['decoded_json_kind'] = field_profile['value_json'].map(decoded_json_kind)
field_profile['json_valid'] = True
type_summary = field_profile['value_type'].value_counts(dropna=False).rename_axis('value_type').reset_index(name='rows')
type_summary['files'] = type_summary['value_type'].map(field_profile.groupby('value_type')['file_id'].nunique())
display(type_summary)

date_mismatches = field_profile.loc[(field_profile['value_type'] == 'date') & (field_profile['decoded_json_kind'] == 'string'), ['file_id', 'key_path', 'value_type', 'value_json']]
display(Markdown('**Discrepancias date vs JSON:** json.loads(value_json) devuelve string para estos valores, lo que es compatible con su serialización JSON y no se corrige en origen.'))
display(date_mismatches)

mixed_type_paths = field_profile.groupby('key_path')['value_type'].nunique().rename('n_value_types').reset_index().query('n_value_types > 1').sort_values('n_value_types', ascending=False)
display(Markdown('**Paths con más de un value_type:** la heterogeneidad se conserva como propiedad del YAML fuente.'))
display(mixed_type_paths.head(30))

field_coverage = field_profile.groupby('root_key')['file_id'].nunique().rename('files_with_field').reset_index().sort_values('files_with_field', ascending=False)
field_coverage['percentage_files'] = (100 * field_coverage['files_with_field'] / workflow_files['file_id'].nunique()).round(2)
display(field_coverage.head(30))

,value_type,rows,files
0,string,22167,1504
1,mapping,17732,1502
2,array,8062,1499
3,boolean,6169,1388
4,integer,4088,1367
5,null,1792,1107
6,float,26,11
7,date,2,2


**Discrepancias date vs JSON:** json.loads(value_json) devuelve string para estos valores, lo que es compatible con su serialización JSON y no se corrige en origen.

,file_id,key_path,value_type,value_json
24361,648a231d73e20870575cfcbb7da4e1388bdf53decf67d3...,ms\.date,date,"""2026-09-04"""
52434,ded40b528382419364241c21472d63c3e4525b61f5698a...,metadata.date,date,"""2026-03-29"""


**Paths con más de un value_type:** la heterogeneidad se conserva como propiedad del YAML fuente.

,key_path,n_value_types
4071,tools.repo-memory,4
4032,tools.cache-memory,4
2989,safe-outputs.create-issue.expires,3
3824,safe-outputs.missing-data,3
6,checkout,3
3830,safe-outputs.noop,3
3981,sandbox.agent,3
4031,tools.bash,3
3826,safe-outputs.missing-tool,3
4048,tools.github,3


,root_key,files_with_field,percentage_files
41,on,1492,96.82
42,permissions,1486,96.43
54,safe-outputs,1418,92.02
65,tools,1379,89.49
5,description,1339,86.89
62,timeout-minutes,1089,70.67
39,network,1024,66.45
8,engine,901,58.47
38,name,725,47.05
19,imports,607,39.39


In [8]:
body_series = workflow_bodies['body_markdown'].fillna('').astype('string')
body_features = pd.DataFrame({
    'file_id': workflow_bodies['file_id'],
    'body_chars': body_series.str.len().fillna(0).astype('int64'),
    'body_words': body_series.str.strip().str.split().str.len().fillna(0).astype('int64'),
})
body_features['is_empty_body'] = body_series.map(lambda value: bool(re.fullmatch(r'\s*', str(value))))
body_stats = pd.DataFrame([
    {'metric': metric, 'value': value}
    for metric, value in {
        'bodies': len(body_features),
        'empty_or_whitespace_bodies': int(body_features['is_empty_body'].sum()),
        'words_min': int(body_features['body_words'].min()),
        'words_max': int(body_features['body_words'].max()),
        'words_mean': round(body_features['body_words'].mean(), 2),
        'words_median': round(body_features['body_words'].median(), 2),
        'chars_min': int(body_features['body_chars'].min()),
        'chars_max': int(body_features['body_chars'].max()),
    }.items()
])
display(body_stats)
display(body_features.loc[body_features['is_empty_body']])

q1, q3 = body_features['body_words'].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
display(pd.DataFrame([{
    'criterion': 'outlier exploratorio de longitud',
    'threshold_words': round(upper_fence, 2),
    'affected_bodies': int((body_features['body_words'] > upper_fence).sum()),
    'treatment': 'Se conservan; solo se considera limitar la vista de gráficos.'
}]))


,metric,value
0,bodies,1541.00
1,empty_or_whitespace_bodies,1.00
2,words_min,0.00
3,words_max,15217.00
4,words_mean,1063.67
5,words_median,750.00
6,chars_min,0.00
7,chars_max,103946.00


,file_id,body_chars,body_words,is_empty_body
462,4a70a663bc4f013bf6d1a01c8927891ed2c9de768c7461...,0,0,True


,criterion,threshold_words,affected_bodies,treatment
0,outlier exploratorio de longitud,2884.5,99,Se conservan; solo se considera limitar la vis...


In [9]:
content_counts = workflow_files.groupby('content_sha256', dropna=False).size()
repeated_content = content_counts[content_counts > 1]
blob_counts = workflow_files.groupby('blob_sha', dropna=False).size()
repeated_blob = blob_counts[blob_counts > 1]
repetition_summary = pd.DataFrame([
    {'hash_column': 'content_sha256', 'repeated_groups': len(repeated_content), 'rows_in_repeated_groups': int(repeated_content.sum()), 'duplicate_extras': int((repeated_content - 1).sum()), 'largest_group': int(repeated_content.max()) if len(repeated_content) else 0},
    {'hash_column': 'blob_sha', 'repeated_groups': len(repeated_blob), 'rows_in_repeated_groups': int(repeated_blob.sum()), 'duplicate_extras': int((repeated_blob - 1).sum()), 'largest_group': int(repeated_blob.max()) if len(repeated_blob) else 0},
])
display(repetition_summary)
display(Markdown('Los hashes repetidos no se eliminan: pueden representar templates o copias y reducen la independencia de algunas observaciones.'))

owner_frame = repositories.assign(owner_key=repositories['owner'].astype('string').str.casefold())
owner_case_groups = owner_frame.groupby('owner_key')['owner'].nunique()
variant_owner_keys = owner_case_groups[owner_case_groups > 1].index
owner_case_rows = int(owner_frame['owner_key'].isin(variant_owner_keys).sum())
display(pd.DataFrame([{
    'control': 'variantes de mayúsculas en owner',
    'affected_owner_groups': int((owner_case_groups > 1).sum()),
    'affected_repository_rows': owner_case_rows,
    'treatment': 'Se conserva el valor original; solo se puede usar una clave casefold para agregaciones auxiliares.'
}]))

,hash_column,repeated_groups,rows_in_repeated_groups,duplicate_extras,largest_group
0,content_sha256,86,192,106,6
1,blob_sha,85,190,105,6


Los hashes repetidos no se eliminan: pueden representar templates o copias y reducen la independencia de algunas observaciones.

,control,affected_owner_groups,affected_repository_rows,treatment
0,variantes de mayúsculas en owner,4,24,Se conserva el valor original; solo se puede u...


# 4. Tratamiento de los problemas encontrados

## 4.1 Decisión de tratamiento

Los controles no muestran evidencia que justifique eliminar o modificar registros estructurales. En consecuencia, no se crea eda/data/processed/: Notebook 2 reconstruye sus tablas derivadas en memoria.

Se documentan las siguientes decisiones:

- Se conservan los cinco repositorios con cero archivos; son observaciones válidas de repositories.
- Se conservan los bodies vacíos y los bodies extremos; se describen y, cuando corresponde, solo se limita la vista de un gráfico.
- Se conservan contenidos repetidos y hashes repetidos; pueden corresponder a templates o copias.
- Se conservan los tipos YAML heterogéneos y los paths escapados.
- No se imputan campos opcionales como engine, tools, permissions o timeout-minutes. Su ausencia no equivale a un error.
- Las validaciones de join y las agregaciones a nivel archivo se realizan antes de calcular estadísticas para evitar multiplicar filas por frontmatter_fields.

In [10]:
# Preparación en memoria para verificar que el tratamiento no altera las tablas originales.
file_frontmatter_counts = frontmatter_fields.groupby('file_id').size().rename('n_frontmatter_fields')
frontmatter_file_summary = workflow_files[['file_id']].drop_duplicates().merge(file_frontmatter_counts, on='file_id', how='left', validate='one_to_one')
frontmatter_file_summary['n_frontmatter_fields'] = frontmatter_file_summary['n_frontmatter_fields'].fillna(0).astype('int64')
frontmatter_file_summary['has_frontmatter'] = frontmatter_file_summary['n_frontmatter_fields'].gt(0)
file_features = workflow_files[['file_id', 'repository_id', 'path', 'parse_status']].merge(body_features, on='file_id', how='left', validate='one_to_one').merge(frontmatter_file_summary, on='file_id', how='left', validate='one_to_one')
assert len(file_features) == workflow_files['file_id'].nunique()
display(file_features.head())
display(Markdown('**Conclusión:** no se escribe ninguna tabla procesada. Estas estructuras se descartan al terminar la ejecución y solo sirven para análisis reproducible en memoria.'))

,file_id,repository_id,path,parse_status,body_chars,body_words,is_empty_body,n_frontmatter_fields,has_frontmatter
0,5defc8d8ae9ce53a2a638b2afd35bf22c48f88ee188243...,002a471c7d3967b4c8cc4d722a1509b7a749c2f2b0219b...,.github/workflows/doc-freshness.md,ok,4034,618,False,25,True
1,a8292f31e77164e648e3007ae74adaea2c5a5af8c08f8a...,002a471c7d3967b4c8cc4d722a1509b7a749c2f2b0219b...,.github/workflows/issue-go-clarify.md,ok,1705,248,False,16,True
2,f65ec881ec083c7bce744ed2a4ca619b6a3cb6e6d8f5be...,002a471c7d3967b4c8cc4d722a1509b7a749c2f2b0219b...,.github/workflows/issue-go-yes.md,ok,3256,471,False,31,True
3,c380a28f500e72841a9e5095c6c951ddfe369ae01885e8...,002a471c7d3967b4c8cc4d722a1509b7a749c2f2b0219b...,.github/workflows/issue-triage.md,ok,3190,475,False,18,True
4,a0fa0445cd7afd149bad8ad0012590cd8c3867d92b8354...,01991a343c445d3b5ec0cac22ffe91d65cf0aea921a522...,.github/workflows/aw-dependabot-pr-review.md,ok,6319,835,False,51,True


**Conclusión:** no se escribe ninguna tabla procesada. Estas estructuras se descartan al terminar la ejecución y solo sirven para análisis reproducible en memoria.